## **1. Configuración del Entorno (Code)**

###*1.1 Importaciones necesarias para trabajar.*

In [63]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
# 1. Habilitar explícitamente las características experimentales
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import f1_score
# Aqui llamamos a nuestro archivo donde quedaron las funciones de limpieza y transformacion.
from src.data_preprocessing import tratar_duplicados, corregir_signos_negativos, ingenieria_caracteristicas, Winsorizer, DataFrameConverter, CorrelationFilter
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LogisticRegression, BayesianRidge

from src.model_evaluation import evaluar_modelo_entrenado

### *1.2 Cargar los registros a la variable "data" con pandas*

In [64]:
ruta_dataset = "../data/data_sucia/dataset_clientes.csv"
data = pd.read_csv(ruta_dataset)

## **2. Preprocesamiento Inicial**

In [65]:
cols_con_negativos = ['ingreso_mensual', 'gasto_mensual', 'deuda_total']

pipeline_limpieza = Pipeline(steps=[
    ('limpieza_duplicados', FunctionTransformer(tratar_duplicados, kw_args={'drop': True})),
    ('correccion_signos', FunctionTransformer(corregir_signos_negativos, kw_args={'columnas': cols_con_negativos})),
    ('creacion_variables', FunctionTransformer(ingenieria_caracteristicas))
])


data_limpio = pipeline_limpieza.fit_transform(data)

X = data_limpio.drop(columns=['id_cliente', 'fecha_registro', 'dia_semana_registro', 'abandono'], errors='ignore')
y = data_limpio['abandono']

# División estratificada de control (Semilla 42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

num_cols = ['ingreso_mensual', 'gasto_mensual', 'score_crediticio', 'deuda_total',
            'ratio_deuda', 'ratio_gasto', 'indice_lealtad', 'riesgo_por_inactividad']
cat_cols = ['genero', 'tipo_plan', 'canal_registro', 'region', 'uso_app', 'estado_civil']

## **3. Canalización Estadística (Pipelines de Transformación)**

In [66]:
# Creamos el estimador interno empaquetado con su propio escalador.
# Esto asegura que cada regresión interna reciba los predictores estandarizados (media 0, var 1),
# eliminando el sesgo por diferencia de escalas en la penalización de BayesianRidge.
estimador_interno = make_pipeline(
    StandardScaler(),
    BayesianRidge()
)

num_pipeline = Pipeline(steps=[
    ('imputacion_iterativa', IterativeImputer(estimator=estimador_interno, random_state=42, max_iter=10)),
    ('tratamiento_outliers', Winsorizer(limits=(0.01, 0.01))),
    ('escalado_estandar', StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ('codificacion_onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
], remainder='drop')

## **4. Modelos Supervisados**

### *4.1 Modelo de Regresión Logística*

In [67]:
resultados_base = {}

In [68]:
pipeline_lr_base = Pipeline(steps=[
    ('duplicados', FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('conversion', DataFrameConverter(preprocessor)),
    ('colinealidad', CorrelationFilter(threshold=0.9)),
    ('clf', LogisticRegression(random_state=42, max_iter=1000, solver='saga'))
])
pipeline_lr_base.fit(X_train, y_train)
resultados_base['Regresión Logística'] = pipeline_lr_base

# Evaluar el modelo
mejores_metricas_lr = evaluar_modelo_entrenado(pipeline_lr_base, X_test, y_test)

print("Metricas de Regresión Logística")
for metrica, valor in mejores_metricas_lr.items():
    print(f"{metrica:<15} : {valor:.4f}")

Metricas de Regresión Logística
accuracy        : 0.6310
precision       : 0.5686
recall          : 0.2899
f1              : 0.3840
roc_auc         : 0.6478


**Evaluacion del Entrenamiento**

In [69]:
print(f"{'Score del modelo en entrenamiento':<40}:{pipeline_lr_base.score(X_train, y_train):.5f}")
print(f"{'Score del modelo en test': <40}:{pipeline_lr_base.score(X_test, y_test):.5f}")

Score del modelo en entrenamiento       :0.63050
Score del modelo en test                :0.63100


### *4.2 Modelo de Árbol de Decisión*

In [70]:
pipeline_dt_base = Pipeline(steps=[
    ('duplicados', FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('conversion', DataFrameConverter(preprocessor)),
    ('colinealidad', CorrelationFilter(threshold=0.9)),
    ('clf', DecisionTreeClassifier(random_state=42))
])
pipeline_dt_base.fit(X_train, y_train)
resultados_base['Árbol de Decisión'] = pipeline_dt_base

# Evaluar el modelo 
mejores_metricas_lr = evaluar_modelo_entrenado(pipeline_dt_base, X_test, y_test)

print("Metricas del Árbol de Decisión")
for metrica, valor in mejores_metricas_lr.items():
    print(f"{metrica:<15} : {valor:.4f}")


Metricas del Árbol de Decisión
accuracy        : 0.5677
precision       : 0.4565
recall          : 0.4694
f1              : 0.4629
roc_auc         : 0.5509


**Evaluacion del Entrenamiento**

In [71]:
print(f"{'Score del modelo en entrenamiento':<40}:{pipeline_dt_base.score(X_train, y_train):.5f}")
print(f"{'Score del modelo en test': <40}:{pipeline_dt_base.score(X_test, y_test):.5f}")

Score del modelo en entrenamiento       :1.00000
Score del modelo en test                :0.56775


### *4.3 Modelo de Máquinas de Vectores de Soporte (SVM)*

In [72]:
pipeline_svm_base = Pipeline(steps=[
    ('duplicados', FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('conversion', DataFrameConverter(preprocessor)),
    ('colinealidad', CorrelationFilter(threshold=0.9)),
    ('clf', SVC(random_state=42, kernel='rbf', probability=True))
])
pipeline_svm_base.fit(X_train, y_train)
resultados_base['SVM'] = pipeline_svm_base

# Evaluar el modelo
mejores_metricas_lr = evaluar_modelo_entrenado(pipeline_svm_base, X_test, y_test)

print("Metricas Maquinas de Vectores de Soporte")
for metrica, valor in mejores_metricas_lr.items():
    print(f"{metrica:<15} : {valor:.4f}")

Metricas Maquinas de Vectores de Soporte
accuracy        : 0.6335
precision       : 0.5819
recall          : 0.2710
f1              : 0.3697
roc_auc         : 0.6306


**Evaluacion del Entrenamiento**

In [73]:
print(f"{'Score del modelo en entrenamiento':<40}:{pipeline_svm_base.score(X_train, y_train):.5f}")
print(f"{'Score del modelo en test': <40}:{pipeline_svm_base.score(X_test, y_test):.5f}")

Score del modelo en entrenamiento       :0.65994
Score del modelo en test                :0.63350


## **5. Evaluación Comparativa de Métricas**

In [74]:
print("F1-SCORE MODELOS LINEA BASE")
for nombre, modelo in resultados_base.items():
    preds = modelo.predict(X_test)
    print(f"{nombre}: {f1_score(y_test, preds):.4f}")

F1-SCORE MODELOS LINEA BASE
Regresión Logística: 0.3840
Árbol de Decisión: 0.4629
SVM: 0.3697


In [75]:
# Variables que fueron eliminadas por presentar colinealidad
pipeline_lr_base.named_steps["colinealidad"].columns_to_drop_

[]

In [76]:
# Variables con las cuales fue entrenado el modelo
pipeline_lr_base.named_steps["conversion"].feature_names_

array(['num__ingreso_mensual', 'num__gasto_mensual',
       'num__score_crediticio', 'num__deuda_total', 'num__ratio_deuda',
       'num__ratio_gasto', 'num__indice_lealtad',
       'num__riesgo_por_inactividad', 'cat__genero_Femenino',
       'cat__genero_Masculino', 'cat__genero_Otro',
       'cat__tipo_plan_Basico', 'cat__tipo_plan_Estandar',
       'cat__tipo_plan_Premium', 'cat__canal_registro_App',
       'cat__canal_registro_Tienda', 'cat__canal_registro_Web',
       'cat__region_Centro', 'cat__region_Norte', 'cat__region_Sur',
       'cat__uso_app_Alto', 'cat__uso_app_Bajo', 'cat__uso_app_Medio',
       'cat__estado_civil_Casado', 'cat__estado_civil_Divorciado',
       'cat__estado_civil_Soltero'], dtype=object)